# 04 — Ray Tracing

Vectors in Space unit, final chapter. Setup, one line to change, what you
should see, going further.

Nothing here is examinable. It is here so that you can watch the vectors
from this unit produce an actual picture.


## Setup

Only `numpy` and `matplotlib`, and numpy is used solely to hold the image.
Every vector below is an ordinary Python tuple, and every operation is one
of the ones from the notes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt


def add(a, b):   return (a[0] + b[0], a[1] + b[1], a[2] + b[2])
def sub(a, b):   return (a[0] - b[0], a[1] - b[1], a[2] - b[2])
def mul(k, a):   return (k * a[0], k * a[1], k * a[2])
def dot(a, b):   return a[0] * b[0] + a[1] * b[1] + a[2] * b[2]
def length(a):   return sqrt(dot(a, a))
def unit(a):     return mul(1.0 / length(a), a)

def cross(a, b):
    return (a[1] * b[2] - a[2] * b[1],
            a[2] * b[0] - a[0] * b[2],
            a[0] * b[1] - a[1] * b[0])


### The scene

A floor at z = 0, one sphere sitting on it, and a light shining from one
fixed direction — like the sun, far enough away that its direction is the
same everywhere.


In [ ]:
EYE      = (0.0, -5.0, 2.2)
TARGET   = (0.0,  0.0, 1.0)         # what the camera aims at
SPHERE_M = (0.0,  0.0, 1.0)
SPHERE_R = 1.0
LIGHT    = unit((0.0, -1.0, 1.0))   # direction TOWARDS the light

EPS = 1e-6                          # see 'shadow acne' in the notes


### The camera

To turn a pixel into a direction you need three perpendicular vectors: one
pointing where the camera looks, one pointing right across the image, and
one pointing up it. Two cross products build all three.

`FOV` is the half-width of the image plane one unit ahead of the eye —
smaller means more zoomed in.


In [ ]:
W, H = 120, 90
FOV  = 0.9

FORWARD = unit(sub(TARGET, EYE))
RIGHT   = unit(cross(FORWARD, (0.0, 0.0, 1.0)))
UP      = cross(RIGHT, FORWARD)          # already a unit vector


def ray_through(i, j):
    """Unit direction from the eye through pixel (i, j)."""
    u = ((i + 0.5) / W * 2 - 1) * FOV
    v = (1 - (j + 0.5) / H * 2) * FOV * H / W
    return unit(add(FORWARD, add(mul(u, RIGHT), mul(v, UP))))


print("forward", tuple(round(c, 3) for c in FORWARD))
print("right  ", tuple(round(c, 3) for c in RIGHT))
print("up     ", tuple(round(c, 3) for c in UP))
print("mutually perpendicular?",
      round(dot(FORWARD, RIGHT), 12),
      round(dot(RIGHT, UP), 12),
      round(dot(UP, FORWARD), 12))


**What you should see.** Three unit vectors and three zeros. The zeros are
the check: a cross product is perpendicular to both its inputs, so RIGHT is
perpendicular to FORWARD by construction and UP is perpendicular to both.

This is the same calculation as finding a normal vector to a plane — here
it is being used to build a coordinate system for the camera.


## Part A — one ray, by hand

The worked example from section 3 of the chapter. The ray leaves the eye
in the direction (0, 1, 0) and meets the sphere.

`hit_sphere` returns the smallest t greater than EPS, or None.


In [ ]:
def hit_sphere(origin, d, M, R):
    w = sub(origin, M)
    b = 2.0 * dot(w, d)
    c = dot(w, w) - R * R
    disc = b * b - 4.0 * c          # a = 1 because d is a unit vector
    if disc < 0:
        return None
    root = sqrt(disc)
    for t in sorted(((-b - root) / 2.0, (-b + root) / 2.0)):
        if t > EPS:
            return t
    return None


def hit_floor(origin, d):
    if abs(d[2]) < EPS:
        return None                 # ray runs parallel to the floor
    t = -origin[2] / d[2]
    return t if t > EPS else None


In [ ]:
origin = (0.0, -5.0, 1.0)          # the eye position used in the notes
d = unit((0.0, 1.0, 0.0))
t = hit_sphere(origin, d, SPHERE_M, SPHERE_R)
P = add(origin, mul(t, d))
n = mul(1.0 / SPHERE_R, sub(P, SPHERE_M))

print("t          ", t)
print("hit point  ", P)
print("unit normal", n)
print("brightness ", max(0.0, dot(n, LIGHT)))


**What you should see.** t = 4.0, the point (0, -1, 1), the normal
(0, -1, 0) and a brightness of about 0.707 — every one of them the number
worked out in the notes.

Change the direction to (0, 1, 0.5) and run it again. The ray now passes
over the top of the sphere and `hit_sphere` returns `None`, so the next
line fails. That failure is the discriminant going negative, which is what
a miss looks like from the inside.


## Part B — the whole image

One ray per pixel. For each one: find the nearest hit, take the normal
there, and apply Lambert's cosine law.

### THE ONE LINE TO CHANGE

`LIGHT` is the direction the light comes from. Try (0, -1, 1) as it
stands, then (1, 0, 0.3) for a low side light, then (0, 0, 1) for a light
directly overhead.


In [ ]:
LIGHT = unit((0.0, -1.0, 1.0))    # <-- change these three numbers

SHADOWS = True


def trace(origin, d):
    """Brightness of whatever this ray sees, between 0 and 1."""
    ts = hit_sphere(origin, d, SPHERE_M, SPHERE_R)
    tf = hit_floor(origin, d)

    if ts is None and tf is None:
        return 0.15                                   # background

    if tf is None or (ts is not None and ts < tf):     # sphere is nearer
        P = add(origin, mul(ts, d))
        n = mul(1.0 / SPHERE_R, sub(P, SPHERE_M))
        albedo = 0.9
    else:                                              # floor is nearer
        P = add(origin, mul(tf, d))
        n = (0.0, 0.0, 1.0)
        albedo = 0.6

    if SHADOWS and hit_sphere(P, LIGHT, SPHERE_M, SPHERE_R) is not None:
        return 0.05                                    # in shadow

    return albedo * max(0.0, dot(n, LIGHT))


img = np.zeros((H, W))
for j in range(H):
    for i in range(W):
        img[j, i] = trace(EYE, ray_through(i, j))

plt.imshow(img, cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()


**What you should see.** A grey sphere sitting on a grey floor, brightest
on the side facing up and towards you, with a small dark crescent of
shadow tucked underneath it.

The shadow is small because the default light is almost directly behind
the camera — which is exactly why photographers avoid lighting from there.

Each piece of mathematics is visible in the picture:

- the **smooth gradient** across the sphere is the cosine, nothing else;
- the **hard edge** of the sphere is the discriminant crossing zero;
- the **shadow** is the second ray finding something in the way;
- the sphere **hides** the floor behind it purely because its t is smaller.

Now change the light and run it again:

- **(1, 0, 0.3)** — a low sun from the right. The sphere splits into a lit
  half and a dark half, and the shadow stretches right across the floor.
  The whole scene dims, because the floor now meets the light at a glancing
  angle: the same cosine, applied to the floor instead of the sphere.
- **(0, 0, 1)** — light directly overhead. The floor reaches its maximum
  brightness, the sphere is lit only on its cap, and the shadow collapses
  to a disc directly beneath it. That is the shadow exercise from the
  chapter, drawn rather than calculated.


## Going further


### G1 — turn the shadows off

Set `SHADOWS = False` in the cell above and render again.


Without shadows the sphere seems to float, and it becomes surprisingly
hard to tell where it is sitting. Shadows are not decoration: they are how
an eye works out where things are relative to one another.

Then find the line `return 0.05` and change it to `return 0.0`. The
shadow becomes pure black, and the image looks worse — real shadows are
never fully dark, because light arrives from elsewhere too.


### G2 — a mirrored floor

When the ray hits the floor, reflect it and trace onwards instead of
stopping. This is the reflection formula from the chapter on vector
geometry, applied once.


In [ ]:
def reflect(d, n):
    return sub(d, mul(2.0 * dot(d, n), n))


def trace_mirror(origin, d, depth=1):
    ts = hit_sphere(origin, d, SPHERE_M, SPHERE_R)
    tf = hit_floor(origin, d)

    if ts is None and tf is None:
        return 0.15

    if tf is None or (ts is not None and ts < tf):
        P = add(origin, mul(ts, d))
        n = mul(1.0 / SPHERE_R, sub(P, SPHERE_M))
        shade = 0.9 * max(0.0, dot(n, LIGHT))
        if hit_sphere(P, LIGHT, SPHERE_M, SPHERE_R) is not None:
            shade = 0.05
        return shade

    P = add(origin, mul(tf, d))
    n = (0.0, 0.0, 1.0)
    shade = 0.6 * max(0.0, dot(n, LIGHT))
    if hit_sphere(P, LIGHT, SPHERE_M, SPHERE_R) is not None:
        shade = 0.05
    if depth > 0:
        shade = 0.6 * shade + 0.4 * trace_mirror(P, reflect(d, n), depth - 1)
    return shade


img = np.zeros((H, W))
for j in range(H):
    for i in range(W):
        img[j, i] = trace_mirror(EYE, ray_through(i, j))

plt.imshow(img, cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()


**What you should see.** The sphere now has a reflection in the floor,
below and slightly distorted.

Raise `depth` to 2 or 3. Almost nothing changes here, because there is
only one mirror — but with two facing mirrors the depth limit is the only
thing stopping the computation from running forever.

Note the line `0.6 * shade + 0.4 * trace_mirror(...)`. A real floor is
partly matte and partly mirrored, and those two numbers are how shiny it
is. Set them to 0.0 and 1.0 for a perfect mirror.


### G3 — your turn

Pick one:

**A second sphere.** Add one with a different centre and radius. You will
have to test both and keep the smaller t — which is the 'nearest hit' rule
from the chapter, now with something to choose between. Do not forget the
shadow test, or one sphere will fail to shade the other.

**An animation.** Put the sphere's centre at p0 + t·v and render one frame
for several values of t, as in the last exercise of the chapter. Read the
parameter as time and you have the beginning of an animation system.

**A prediction first.** Before running anything, sketch on paper where you
expect the shadow to fall for LIGHT = (1, 0, 0.3). Then render it. If the
picture disagrees with your sketch, one of the two is wrong, and finding
out which is the exercise.
